In [1]:
# Install required packages
!pip install -q transformers>=4.40.0 datasets>=2.16.0 accelerate>=0.25.0 peft>=0.7.0 bitsandbytes>=0.41.0
!pip install -q torch>=2.0.0 scipy>=1.11.0 scikit-learn>=1.3.0
!pip install -q wandb tqdm

# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU Available: True
GPU: Tesla T4
GPU Memory: 15.8 GB


In [2]:
import json
import os
import gc
from typing import Dict, List, Any
from dataclasses import dataclass
from datetime import datetime
import subprocess

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
import wandb
from tqdm.auto import tqdm
import numpy as np

In [4]:
# Load the training data
def load_training_data(file_path: str) -> List[Dict[str, Any]]:
    """Load training data from JSON file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"Loaded {len(data)} training examples")

    # Show sample
    print("\nSample training example:")
    print(json.dumps(data[0], indent=2))

    return data

def split_data(data: List[Dict[str, Any]], train_ratio: float = 0.8):
    """Split data into train/validation sets."""
    train_size = int(train_ratio * len(data))
    train_data = data[:train_size]
    val_data = data[train_size:]

    print(f"\nTrain examples: {len(train_data)}")
    print(f"Validation examples: {len(val_data)}")

    return train_data, val_data

# Load and split the data
training_data = load_training_data('training_data_100_examples.json')
train_data, val_data = split_data(training_data)

Loaded 100 training examples

Sample training example:
{
  "input": "How many heads of the publishers are older than 56 ?",
  "output": {
    "intent": "intents_discovery",
    "discovery_results": [
      {
        "step_id": "step_1",
        "sub_question": "How many heads of the publishers are older than 56 ?",
        "measures": [
          {
            "name": "Heads",
            "calculation": "Count",
            "original_phrase": "how many heads"
          }
        ],
        "dimensions": [
          {
            "name": "Publisher",
            "filter_value": null,
            "original_phrase": "publishers"
          },
          {
            "name": "Age",
            "filter_value": "older than 56",
            "original_phrase": "older than 56"
          }
        ],
        "timegrain": null,
        "timeframe": null,
        "pattern": null,
        "segments": [],
        "breakdowns": [],
        "unmatched_intents": []
      }
    ]
  }
}

Train examples: 8

In [5]:
def load_prompt_template() -> str:
    """Load the prompt template for BI intent discovery."""
    prompt_template = """# Planning and Discovery Agent
You are an AI assistant specialized in analyzing natural language questions about business intelligence data and breaking them down into structured steps for query building.

## Your Role
You have two main phases of operation:
### Phase 1: Planning (for complex questions)
- Analyze the user's question to determine if it requires multi-step processing
- For complex questions, break them down into structured steps that can be used to build CTEs (Common Table Expressions)
- Identify dependencies between steps
- For simple questions, skip this phase and go directly to discovery

### Phase 2: Discovery
- Analyze the question (or planning steps) to identify BI concepts
- Map natural language terms to specific dimensions, measures, and filters
- Handle ambiguity by requesting clarification when needed

## Question Complexity Assessment
A question is COMPLEX if it contains ANY of these logical patterns:
1. **Implicit Dependencies**: When one concept depends on another
2. **Sequential Logic**: When steps must be performed in order
3. **Ranking/Selection Logic**: When filtering requires prior analysis
4. **Multi-Step Filtering**: When filters depend on other filters
5. **Comparative Analysis**: When comparing requires separate data gathering
6. **Time-Based Dependencies**: When time periods affect other queries

## Response Format
Respond with a JSON object containing your intent and the appropriate data structure based on the phase you're executing.

## Current Context
- User Question: {question}

## Instructions
1. Assess Question Complexity: Determine if this is a simple or complex question
2. For Complex Questions: Execute planning phase first, then discovery phase on the planning steps
3. For Simple Questions: Skip planning phase, execute discovery phase directly on the question
4. Handle Ambiguity: Request human input when terms are unclear
5. Use Available Tools: Use the appropriate tool based on your phase and intent

## Response Format
Respond with a JSON object containing your intent and the appropriate data structure based on the phase you're executing.

Output:"""

    return prompt_template

# Load prompt template
prompt_template = load_prompt_template()
print("Prompt template loaded successfully")

Prompt template loaded successfully


In [6]:
@dataclass
class TrainingExample:
    """Represents a single training example."""
    question: str
    expected_output: Dict[str, Any]

class BIIntentDataset(Dataset):
    """Dataset for BI Intent Discovery training."""

    def __init__(self, data: List[Dict[str, Any]], tokenizer, max_length: int = 2048):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]

        # Create the input prompt
        input_text = prompt_template.format(question=example['input'])

        # Create the expected output
        output_text = json.dumps(example['output'], ensure_ascii=False, separators=(',', ':'))

        # Combine input and output
        full_text = input_text + output_text

        # Tokenize WITHOUT padding - let the collator handle it
        encoding = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding=False,  # Changed from True to False
            return_tensors=None  # Changed from 'pt' to None
        )

        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': encoding['input_ids'].copy()
        }

print("Dataset class created")

Dataset class created


In [7]:
def load_model_and_tokenizer(model_name: str = "Qwen/Qwen2.5-0.5B"):
    """Load Qwen model and tokenizer with 4-bit quantization."""
    print(f"Loading model: {model_name}")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"Tokenizer loaded")
    print(f"Vocabulary size: {tokenizer.vocab_size}")
    print(f"Special tokens: {tokenizer.special_tokens_map}")

    # Load model with 4-bit quantization for memory efficiency
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        quantization_config={
            "load_in_4bit": True,
            "bnb_4bit_compute_dtype": torch.float16,
            "bnb_4bit_use_double_quant": True,
            "bnb_4bit_quant_type": "nf4"
        }
    )

    print(f"Model loaded with 4-bit quantization")
    print(f"Model parameters: {model.num_parameters():,}")

    return model, tokenizer

# Load model and tokenizer
model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-0.5B")

Loading model: Qwen/Qwen2.5-0.5B


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded
Vocabulary size: 151643
Special tokens: {'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded with 4-bit quantization
Model parameters: 494,032,768


In [8]:
def setup_lora(model):
    """Setup LoRA for efficient fine-tuning."""
    # Prepare model for training
    model = prepare_model_for_kbit_training(model)

    # LoRA configuration
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=16,  # Rank
        lora_alpha=32,  # Alpha parameter
        lora_dropout=0.1,  # Dropout
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    # Apply LoRA
    model = get_peft_model(model, lora_config)

    print("LoRA applied successfully")
    model.print_trainable_parameters()

    # Enable gradient checkpointing for memory efficiency
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

    return model, lora_config

# Setup LoRA
model, lora_config = setup_lora(model)

LoRA applied successfully
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
Gradient checkpointing enabled


In [9]:
def create_datasets(train_data, val_data, tokenizer, max_length: int = 2048):
    """Create training and validation datasets."""
    train_dataset = BIIntentDataset(train_data, tokenizer, max_length)
    val_dataset = BIIntentDataset(val_data, tokenizer, max_length)

    print(f"Training dataset created: {len(train_dataset)} examples")
    print(f"Validation dataset created: {len(val_dataset)} examples")

    # Test a sample
    sample = train_dataset[0]
    print(f"\nSample input length: {len(sample['input_ids'])} tokens")

    # Decode sample to verify
    sample_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    print(f"\nSample decoded (first 500 chars):")
    print(sample_text[:500] + "...")

    return train_dataset, val_dataset

# Create datasets
train_dataset, val_dataset = create_datasets(train_data, val_data, tokenizer, 2048)

Training dataset created: 80 examples
Validation dataset created: 20 examples

Sample input length: 554 tokens

Sample decoded (first 500 chars):
# Planning and Discovery Agent
You are an AI assistant specialized in analyzing natural language questions about business intelligence data and breaking them down into structured steps for query building.

## Your Role
You have two main phases of operation:
### Phase 1: Planning (for complex questions)
- Analyze the user's question to determine if it requires multi-step processing
- For complex questions, break them down into structured steps that can be used to build CTEs (Common Table Expressi...


In [10]:
def create_training_args(output_dir: str = "./qwen-bi-intent-model",
                        num_epochs: int = 3,
                        batch_size: int = 2,
                        learning_rate: float = 2e-4):
    """Create training arguments."""
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8
        warmup_steps=10,
        learning_rate=learning_rate,
        fp16=True,  # Use mixed precision
        logging_steps=5,
        eval_strategy="steps",  # Changed from evaluation_strategy
        eval_steps=20,
        save_steps=20,  # Changed from 50 to 20 (must be multiple of eval_steps)
        save_total_limit=2,  # Keep only 2 checkpoints
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="wandb" if wandb.run else None,  # Optional: wandb logging
        dataloader_pin_memory=False,  # Save memory
        remove_unused_columns=False,
        push_to_hub=False,  # Set to True if you want to push to Hugging Face Hub
    )

    print("Training arguments configured")
    print(f"Total training steps: ~{num_epochs * 40}")  # Approximate
    print(f"Estimated training time: ~10-15 minutes")

    return training_args

# Create training arguments
training_args = create_training_args(
    num_epochs=3,
    batch_size=2,
    learning_rate=2e-4
)

Training arguments configured
Total training steps: ~120
Estimated training time: ~10-15 minutes


In [11]:
def initialize_training(model, tokenizer, train_dataset, val_dataset, training_args):
    """Initialize the trainer."""
    # Initialize wandb (optional)
    if wandb.run is None:
        wandb.init(
            project="qwen-bi-intent-discovery",
            name=f"qwen2.5-0.5b-bi-intent-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
            config={
                "model": "Qwen/Qwen2.5-0.5B",
                "dataset_size": len(train_dataset) + len(val_dataset),
                "max_length": 2048,
                "batch_size": training_args.per_device_train_batch_size,
                "learning_rate": training_args.learning_rate,
                "epochs": training_args.num_train_epochs
            }
        )

    # Create a custom data collator with padding
    def custom_data_collator(features):
        # Find the maximum length in this batch
        max_length = max(len(f['input_ids']) for f in features)

        # Pad all sequences to the same length
        padded_input_ids = []
        padded_attention_mask = []
        padded_labels = []

        for f in features:
            # Pad input_ids
            input_ids = f['input_ids'] + [tokenizer.pad_token_id] * (max_length - len(f['input_ids']))
            padded_input_ids.append(input_ids)

            # Pad attention_mask
            attention_mask = f['attention_mask'] + [0] * (max_length - len(f['attention_mask']))
            padded_attention_mask.append(attention_mask)

            # Pad labels
            labels = f['labels'] + [-100] * (max_length - len(f['labels']))
            padded_labels.append(labels)

        batch = {}
        batch['input_ids'] = torch.tensor(padded_input_ids)
        batch['attention_mask'] = torch.tensor(padded_attention_mask)
        batch['labels'] = torch.tensor(padded_labels)
        return batch

    # Create trainer with custom data collator
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=custom_data_collator,
    )

    return trainer

# Initialize trainer
trainer = initialize_training(model, tokenizer, train_dataset, val_dataset, training_args)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sukruthisantosh (sukruthisantosh-imperial-college-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipython-input-737621746.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
def train_model(trainer):
    """Start the training process."""
    print("🚀 Starting training...")
    print("⏱️ This will take approximately 10-15 minutes")

    # Start training
    trainer.train()

    print("Training completed!")
    return trainer

# Start training
trainer = train_model(trainer)

🚀 Starting training...
⏱️ This will take approximately 10-15 minutes


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
20,0.322600,0.197264


Training completed!


In [13]:
def save_model(trainer, model_path: str = "./qwen-bi-intent-model-initial"):
    """Save the trained model and configuration."""
    # Save the model
    trainer.save_model(model_path)
    trainer.tokenizer.save_pretrained(model_path)

    print(f"Model saved to: {model_path}")

    # Save training config
    config = {
        "model_name": "Qwen/Qwen2.5-0.5B",
        "training_data_size": len(trainer.train_dataset) + len(trainer.eval_dataset),
        "max_length": 2048,
        "batch_size": trainer.args.per_device_train_batch_size,
        "learning_rate": trainer.args.learning_rate,
        "epochs": trainer.args.num_train_epochs,
        "training_date": datetime.now().isoformat()
    }

    with open(f"{model_path}/training_config.json", 'w') as f:
        json.dump(config, f, indent=2)

    print(f"Training config saved")

    return model_path

# Save the model
model_path = save_model(trainer)

# Download the model
from google.colab import files
import shutil

# Create a zip file of the model
shutil.make_archive('qwen-bi-intent-model', 'zip', './qwen-bi-intent-model-initial')

# Download the zip file
files.download('qwen-bi-intent-model.zip')

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Model saved to: ./qwen-bi-intent-model-initial
Training config saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
def analyze_training_results(trainer):
    """Analyze and display training results."""
    # Get training history
    history = trainer.state.log_history

    if not history:
        print("No training history available")
        return

    print("Training Results Analysis")
    print("=" * 50)

    # Extract metrics
    train_losses = [log['loss'] for log in history if 'loss' in log]
    eval_losses = [log['eval_loss'] for log in history if 'eval_loss' in log]

    if train_losses:
        print(f"Training Loss:")
        print(f"   Start: {train_losses[0]:.4f}")
        print(f"   End: {train_losses[-1]:.4f}")
        print(f"   Improvement: {train_losses[0] - train_losses[-1]:.4f}")

    if eval_losses:
        print(f"\n Validation Loss:")
        print(f"   Start: {eval_losses[0]:.4f}")
        print(f"   End: {eval_losses[-1]:.4f}")
        print(f"   Best: {min(eval_losses):.4f}")

    # Check for overfitting
    if train_losses and eval_losses:
        final_train_loss = train_losses[-1]
        final_eval_loss = eval_losses[-1]

        if final_eval_loss > final_train_loss * 1.2:
            print("\n Potential overfitting detected (validation loss > 1.2x training loss)")
        else:
            print("\n No significant overfitting detected")

    print(f"\n🎯 Recommendations:")
    if eval_losses and eval_losses[-1] > 2.0:
        print("   • Consider more training epochs")
        print("   • Try different learning rate")
        print("   • Increase dataset size")
    else:
        print("   • Model training looks good!")
        print("   • Consider testing on more examples")

# Analyze training results
analyze_training_results(trainer)

Training Results Analysis
Training Loss:
   Start: 2.3684
   End: 0.1947
   Improvement: 2.1737

 Validation Loss:
   Start: 0.1973
   End: 0.1973
   Best: 0.1973

 No significant overfitting detected

🎯 Recommendations:
   • Model training looks good!
   • Consider testing on more examples


In [ ]:
import time

def load_trained_model(model_path: str):
    """Load the trained model for inference."""
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    return model, tokenizer

def generate_response(model, tokenizer, question: str, max_new_tokens: int = 512):
    """Generate response for a given question."""
    # Create input prompt
    input_text = prompt_template.format(question=question)

    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate with timing
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,  # Low temperature for consistent outputs
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.time()
    inference_time = end_time - start_time

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the generated part (after the prompt)
    response = generated_text[len(input_text):].strip()

    return response, inference_time

def test_model(model_path: str = "./qwen-bi-intent-model-initial"):
    """Test the trained model on sample questions."""
    # Test questions
    test_questions = [
        "How many publishers have revenue above $1M?",
        "What is the average age of campaign managers?",
        "Show me the top 5 performing campaigns by engagement rate",
        "Compare revenue between Q1 and Q2 for all publishers"
    ]

    print("Testing the trained model...")

    try:
        model, tokenizer = load_trained_model(model_path)

        for i, question in enumerate(test_questions, 1):
            print(f"\nTest {i}: {question}")
            print("-" * 50)

            response, inference_time = generate_response(model, tokenizer, question)
            print(f"Inference time: {inference_time:.3f} seconds")
            print(f"Response:")
            print(response)

            # Try to parse as JSON
            try:
                parsed = json.loads(response)
                print("Valid JSON output")
            except json.JSONDecodeError:
                print("Invalid JSON output")

            print("=" * 50)

    except Exception as e:
        print(f"Error loading model: {e}")
        print("Try running the training cell again or check the model path")

# Test the model
test_model(model_path)

Testing the trained model...

Test 1: How many publishers have revenue above $1M?
--------------------------------------------------
Inference time: 31.521 seconds
Response:
{"intent":"intents_discovery","discovery_results":[{"step_id":"step_1","sub_question":"How many publishers have revenue above $1M?","measures":[{"name":"Publishers","filter_value":null,"original_phrase":"publishers"},{"name":"Revenue","filter_value":null,"original_phrase":"revenue"},{"name":"Above $1M","filter_value":null,"original_phrase":"above $1M"}],"dimensions":[{"name":"Publisher","filter_value":null,"original_phrase":"publishers"}],"timegrain":null,"timeframe":null,"pattern":null,"pattern_type":null,"pattern_phrase":null,"pattern_range":null,"pattern_range_phrase":null,"pattern_range_range":null,"pattern_range_range_phrase":null,"pattern_range_range_range":null,"pattern_range_range_range_phrase":null,"pattern_range_range_range_range":null,"pattern_range_range_range_range_phrase":null,"pattern_range_range_ran